# Data Layer Workflow

This notebook shows how controlled fund, position, investor, and market-data inputs are loaded into the project database and prepared for downstream fund risk workflows, such as VaR, Expected Shortfall, stress testing, liquidity monitoring, leverage monitoring, board reporting, and Annex IV reporting.

Objectives:

1. Show how the repository builds a structured data layer from controlled source files.
2. Show how table-level SQL checks used in many risk workflows are translated into reusable Python helper functions.

This is not a risk report. It is a data-layer workflow notebook. 

> Note 1: Imports are placed close to their first use to keep each workflow section self-contained.

> Note 2: SQL examples are included to make the database layer familiar for users who usually work directly with tables. They are not the main interface used by the workflow notebooks.

## 1. Workflow scope

The workflow has four layers:

1. **Source data layer**

   Controlled JSON inputs under [`reference_data/`](reference_data) and generated position files under [`data/`](data/).

2. **SQLite database layer**

   Local SQLite database created by [`setup_db.py`](src/setup_db.py). It stores fund profiles, investor records, and positions in structured tables. The database is used to simulate the type of SQL data layer commonly found in risk workflows, where analysts inspect, reconcile, and query data before using it in downstream calculations.

3. **Market data and enrichment layer**

   Bloomberg-style market-data interface using `bdp()` for point-in-time fields and `bdh()` for historical series. Enrichment adds market-data fields and risk sensitivities to raw position data.

4. **Analytics input layer**

   Risk-ready DataFrame used by downstream workflows for VaR, stress testing, liquidity, leverage, board reporting, and regulatory reporting.

## 2. Source data layer

The repository uses controlled inputs under [`reference_data/`](reference_data) as the starting point for the example dataset. The source data layer is organised into three top-level areas:

- [`reference_data/platform/`](reference_data/platform/) contains platform-wide files shared across all funds, such as the fund registry, fund file map, and investor type mapping.
- [`reference_data/regulation/`](reference_data/regulation/) contains central regulatory configuration files for prescribed values that apply where regulation sets a specific rule or limit.

Fund-level reference files are stored under `reference_data/funds/<fund_name>/`. The main files are:

- Each `fund_profile.json` stores static fund profile information used to build the `funds` table.
- Each `risk_policy.json` stores fund-specific risk parameters, such as board-approved thresholds, redemption scenarios, market scenarios, liquidity assumptions, and VaR multipliers, where the value is set at fund level.
- Position specification files define reproducible holdings used to generate position files.

> Note: The JSON input structures and loader validation rules are documented in [`docs/schemas/reference_data_schema.md`](docs/schemas/reference_data_schema.md).

## 3. Database setup

The SQLite database is created locally through `setup_db.py` for this example. The setup workflow creates the database structure, loads controlled reference records, and populates the raw position tables from generated position files. The database is stored at `data/risk_management.db`. In production, the database would normally already exist, and scheduled data workflows would update it daily. In this notebook, the setup step is run explicitly so the example remains reproducible.

In [ ]:
from src.setup_db import run as setup_db
setup_db(force=True)

from src.data.database import get_engine
engine = get_engine()

## 4. Database schema

The SQLite database contains tables for fund metadata and position-based risk workflows.

| Group | Tables | Purpose |
|---|---|---|
| Fund metadata | `funds` | Static fund metadata loaded from `fund_registry.json` and each fund's `fund_profile.json` |
| Position-based model | `positions`, `positions_enriched`, `instruments` | Raw and enriched holdings used by the risk workflows |
| Time series | NAV and market-data tables where available | Historical series used by performance, P&L, and risk workflows |


## 5. SQL inspection of the fund-risk database

This section uses SQL because many users understand database workflows through table-level queries. It is the only notebook where the SQL layer is shown directly. In the other project notebooks, database queries for positions, enriched data, and platform-level records are handled inside reusable Python functions, so the notebook code stays focused on risk analysis and results rather than data querying.

### 5.1 Platform Level Queries

#### 5.1.1 Available funds

Start by checking which funds are loaded into the database.

In [ ]:
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format
from sqlalchemy import text

funds = pd.read_sql(
    text(
        """
        SELECT
            fund_id,
            fund_name,
            currency,
            fund_type
        FROM funds
        ORDER BY fund_id
        """
    ),
    engine,
)

funds

#### 5.1.2 Table inventory

List available database tables before querying positions or enriched data.

In [ ]:
tables = pd.read_sql(
    text(
        """
        SELECT name AS table_name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """
    ),
    engine,
)

tables

### 5.1 Fund Level Queries

- `FUND_ID` identifies the selected fund for fund-specific checks.
- `VALUATION_DATE` identifies the as-of date for position, market-data, and risk-input checks.

The project uses controlled mock data with a fixed valuation date of `2026-03-31` so the examples are reproducible.

In [ ]:
from src.config import VALUATION_DATE
FUND_ID = "AIFM_HedgeFund"

Inspect the selected fund record before loading positions.

In [ ]:
fund_metadata = pd.read_sql(
    text(
        """
        SELECT *
        FROM funds
        WHERE fund_id = :fund_id
        """
    ),
    engine,
    params={"fund_id": FUND_ID},
)

fund_metadata

### 5.4 Position table schema

Review the position table structure before querying position records.

In [ ]:
position_schema = pd.read_sql(
    text(
        """
        PRAGMA table_info(positions)
        """
    ),
    engine,
)

position_schema[["name", "type", "notnull", "pk"]]

### 5.5 Position snapshot check

Query the selected fund and valuation date directly in SQL. This check reconciles position count, NAV, long exposure, short exposure, and gross exposure before the same data is accessed through Python helper functions.

In [ ]:
position_snapshot_check = pd.read_sql(
    text(
        """
        SELECT
            fund_id,
            position_date,
            COUNT(*) AS position_count,
            SUM(market_value_eur) AS nav_eur,
            SUM(CASE WHEN market_value_eur > 0 THEN market_value_eur ELSE 0 END) AS long_exposure_eur,
            SUM(CASE WHEN market_value_eur < 0 THEN market_value_eur ELSE 0 END) AS short_exposure_eur,
            SUM(ABS(market_value_eur)) AS gross_exposure_eur
        FROM positions
        WHERE fund_id = :fund_id
          AND position_date = :position_date
        GROUP BY fund_id, position_date
        """
    ),
    engine,
    params={"fund_id": FUND_ID, "position_date": VALUATION_DATE},
)

position_snapshot_check

### 5.6 Asset-class exposure check

Aggregate the selected fund's valuation-date positions by asset class directly in SQL. This check gives a quick view of position count, net market value, and gross market value.

In [ ]:
asset_class_exposure = pd.read_sql(
    text(
        """
        SELECT
            asset_class,
            COUNT(*) AS position_count,
            SUM(market_value_eur) AS net_market_value_eur,
            SUM(ABS(market_value_eur)) AS gross_market_value_eur
        FROM positions
        WHERE fund_id = :fund_id
          AND position_date = :position_date
        GROUP BY asset_class
        ORDER BY gross_market_value_eur DESC
        """
    ),
    engine,
    params={"fund_id": FUND_ID, "position_date": VALUATION_DATE},
)

asset_class_exposure

## 6. Python helper functions for repeated data access

The previous section shows the SQL layer directly so the database structure is clear. This section presents helper functions from `src.data.database` that run the required queries internally. This keeps later notebooks focused on risk analysis and results while keeping the database logic available in one place.

### 6.1 Query positions through the database helper

The `query_positions()` helper filters the `positions.position_date` column to retrieve a snapshot for a specific date.


In [ ]:
from src.data.database import query_positions
positions = query_positions(engine, FUND_ID, VALUATION_DATE)
positions.head()

### 6.2 Query historical NAV
```
The `query_nav_history` helper wraps the following query:

sql = text(
        'SELECT position_date as date, SUM(market_value_eur) as nav_eur '
        'FROM positions '
        'WHERE fund_id = :fund_id '
        'GROUP BY position_date '
        'ORDER BY position_date'
    )
```

In [ ]:
from src.data.database import query_nav_history
nav_history = query_nav_history(engine, FUND_ID)
nav_history["date"] = pd.to_datetime(nav_history["date"])
nav_history.tail()

## 7. Market data interface

The repository includes a lightweight market-data interface that mirrors the type of workflow normally handled through Bloomberg or another market-data vendor.

The interface exposes Bloomberg-style methods:

- `bdp()` for point-in-time fields
- `bdh()` for historical series

It also uses Bloomberg-style field names, including:

- `PX_LAST`
- `BETA`
- `DUR_ADJ_MID`
- `CONVEXITY`
- `YLD_YTM_MID`

In this project, the interface is backed by controlled mock data, `yfinance` where appropriate, and local caching for downloaded time series. This keeps the examples reproducible while keeping analytics code independent from the underlying data source.

The same workflow can therefore use the mock interface in this repository and a real vendor adapter in a production setting.

In [ ]:
from src.data.mock_bloomberg import MockBloomberg
bbg = MockBloomberg(seed=42)
type(bbg)

### 7.1 `bdp`: point-in-time market-data fields

Use `bdp()` to retrieve fields such as beta, duration, yield, rating, currency, and average trading volume for one or more securities.

In [ ]:
equity_fields = ["NAME", "CRNCY", "BETA", "VOLUME_AVG_20D"]
equity_bdp = bbg.bdp("AAPL US Equity", equity_fields)
equity_bdp

In [ ]:
bond_fields = ["NAME", "CRNCY", "DUR_ADJ_MID", "CONVEXITY", "YLD_YTM_MID", "RTG_SP"]
bond_bdp = bbg.bdp("US912828YK09 Govt", bond_fields)
bond_bdp

### 7.2 `bdh`: historical market-data series

Use `bdh()` to retrieve historical series for backtesting, historical VaR, market context, and price-move checks.

In [ ]:
price_history = bbg.bdh(
    "AAPL US Equity",
    "PX_LAST",
    "2026-03-01",
    VALUATION_DATE,
)

price_history.tail()

<small>
Historical market data downloaded through the mock market-data interface may be cached locally under <code>data/yf_cache/</code>. The cache is a development convenience and can be refreshed when the requested date range is not covered.
</small>

## 8. Enrichment

Raw positions from the database include quantity, price, market value, and fund administrator fields. They do not always include market-data fields or risk sensitivities such as beta, duration, convexity, yield, or average trading volume.

Enrichment adds these fields for liquid instruments and stores the enriched records in the `positions_enriched` table.

| Asset class | Example enriched fields | Source |
|---|---|---|
| Equity | `beta`, `eqy_dvd_yld_ind`, `volume_avg_20d` | Bloomberg-style market data |
| Bond | `dur_adj_mid`, `convexity`, `yld_ytm_mid`, `rtg_sp` | Bloomberg-style market data |
| FX | `px_last` | Bloomberg-style market data |
| Derivative | `delta`, `convexity`, underlying price | Bloomberg-style market data where applicable |

In [ ]:
position_based_funds = pd.read_sql(
    text(
        """
        SELECT fund_id
        FROM funds
        WHERE fund_id NOT IN ('AIFM_PE_Buyout', 'AIFM_Infra_Core')
        ORDER BY fund_id
        """
    ),
    engine,
)

position_based_funds

In [ ]:
from src.data.enrichment import enrich_positions
for fund_row in position_based_funds.itertuples(index=False):
    enrich_positions(engine, fund_id=fund_row.fund_id, position_date=VALUATION_DATE)

### 8.1 Enriched table check

After running the enrichment workflow, inspect the `positions_enriched` table directly with SQL.

This check confirms that enriched records were written for the selected fund and valuation date, and shows coverage for key risk fields such as beta, duration, and convexity.

In [ ]:
enriched_check = pd.read_sql(
    text(
        """
        SELECT
            fund_id,
            position_date,
            COUNT(*) AS enriched_positions,
            SUM(CASE WHEN beta IS NOT NULL THEN 1 ELSE 0 END) AS beta_available,
            SUM(CASE WHEN dur_adj_mid IS NOT NULL THEN 1 ELSE 0 END) AS duration_available,
            SUM(CASE WHEN convexity IS NOT NULL THEN 1 ELSE 0 END) AS convexity_available
        FROM positions_enriched
        WHERE fund_id = :fund_id
          AND position_date = :position_date
        GROUP BY fund_id, position_date
        """
    ),
    engine,
    params={"fund_id": FUND_ID, "position_date": VALUATION_DATE},
)

enriched_check

## 9. Analytics input layer

`get_risk_ready_df()` loads enriched position data into a pandas DataFrame for downstream risk workflows.

```text
Data layer
  ├─ raw positions
  ├─ fund metadata
  ├─ NAV history
  └─ enriched position fields
      ↓
Risk-ready DataFrame
      ↓
Computation modules
  ├─ historical VaR and Expected Shortfall
  ├─ stress testing
  ├─ liquidity monitoring
  ├─ leverage monitoring
  └─ P&L attribution
      ↓
Board reporting, Annex IV, and fund-monitoring notebooks
```

In [ ]:
from src.data.enrichment import enrich_positions, get_risk_ready_df
risk_df = get_risk_ready_df(engine, FUND_ID, VALUATION_DATE)

key_cols = ['isin', 'instrument_name', 'asset_class', 'market_value_eur', 'beta', 'dur_adj_mid', 'currency']
risk_df[key_cols].head()